# 2-bob. 2-BOB. KIBERHUJUMLAR VA XAVFSIZLIK MA'LUMOTLARI
## Kengaytirilgan amaliy ish: provenans grafida hujum zanjirini tiklash

[![Colab'da ochish](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mrmsi/si-kiberxavfsizlik/blob/main/02-bob/2-2_provenans-grafida-hujum-zanjirini-tiklash.ipynb)

*"Sun'iy intellektni kiberxavfsizlikda qo'llash" darsligi, 2-bob.*

> Bu kengaytirilgan amaliy ish darslikning **60610700 – Sun'iy intellekt yo'nalishi versiyasida** keltirilgan.

Quyidagi dastur Sysmon jurnaliga o'xshash bir kunlik hodisalardan provenans grafini quradi. So'ngra u uchta vazifani bajaradi: odatiy faoliyatda deyarli uchramagan "ota → bola" jarayon juftliklarini topadi, shubhali tashqi ulanishdan orqaga kuzatish orqali hujum boshlangan faylni aniqlaydi va oldinga kuzatish orqali zararli hujjat ta'sir qilgan barcha obyektlarni sanab beradi. Dasturda graflar bilan ishlash uchun networkx kutubxonasidan foydalaniladi.

**Ishga tushirish:** yuqoridagi menyudan *Runtime → Run all* (*Ishga tushirish → Hammasini bajarish*) ni tanlang yoki kataklarni birma-bir Shift+Enter bilan bajaring. Barcha ma'lumotlar dasturning o'zida yaratiladi, qo'shimcha fayl yuklash shart emas.

In [1]:
import networkx as nx
from collections import Counter

In [2]:
# 1. Sysmon jurnaliga o'xshash hodisalar: (vaqt, amal, manba, nishon)
#    Odatiy faoliyat: 30 kun davomida takrorlanadigan "ota -> bola" jarayon juftliklari
odatiy_juftlar = Counter({("explorer.exe", "outlook.exe"): 300, ("explorer.exe", "winword.exe"): 250,
                          ("explorer.exe", "chrome.exe"): 900, ("chrome.exe", "chrome.exe"): 5000,
                          ("services.exe", "svchost.exe"): 3000, ("outlook.exe", "winword.exe"): 120,
                          ("explorer.exe", "powershell.exe"): 15})

bugun = [
    ("09:02", "ishga_tushirdi", "explorer.exe", "outlook.exe"),
    ("09:05", "yaratdi",        "outlook.exe",  "C:/Users/aziz/Downloads/hisobot.docm"),
    ("09:05", "ishga_tushirdi", "outlook.exe",  "winword.exe"),
    ("09:05", "o'qidi",         "winword.exe",  "C:/Users/aziz/Downloads/hisobot.docm"),
    ("09:06", "ishga_tushirdi", "winword.exe",  "powershell.exe"),
    ("09:06", "ulandi",         "powershell.exe", "198.51.100.23:443"),
    ("09:06", "yaratdi",        "powershell.exe", "C:/Users/aziz/AppData/upd.exe"),
    ("09:07", "ishga_tushirdi", "powershell.exe", "upd.exe"),
    ("09:07", "ulandi",         "upd.exe",      "203.0.113.50:8443"),
    ("09:30", "ishga_tushirdi", "explorer.exe", "chrome.exe"),
    ("09:31", "ulandi",         "chrome.exe",   "192.0.2.10:443"),
]

In [3]:
# 2. Provenans grafi: tugun - obyekt, qirra - amal (vaqt bilan)
G = nx.MultiDiGraph()
#    "o'qidi" amalida axborot fayldan jarayonga oqadi, shuning uchun qirra teskari yo'naltiriladi
for vaqt, amal, manba, nishon in bugun:
    if amal == "o'qidi":
        G.add_edge(nishon, manba, amal=amal, vaqt=vaqt)
    else:
        G.add_edge(manba, nishon, amal=amal, vaqt=vaqt)

In [4]:
# 3. Noyob "ota -> bola" juftliklari: odatiy faoliyatda kam uchragan yoki umuman uchramaganlar
jami = sum(odatiy_juftlar.values())
print("Shubhali jarayon juftliklari:")
for vaqt, amal, manba, nishon in bugun:
    if amal == "ishga_tushirdi":
        soni = odatiy_juftlar[(manba, nishon)]
        if soni / jami < 0.001:
            print(f"  {vaqt}  {manba} -> {nishon}   (30 kunda {soni} marta)")

Shubhali jarayon juftliklari:
  09:06  winword.exe -> powershell.exe   (30 kunda 0 marta)
  09:07  powershell.exe -> upd.exe   (30 kunda 0 marta)


In [5]:
# 4. Orqaga kuzatish: shubhali tashqi ulanish qayerdan boshlangan?
nishon = "203.0.113.50:8443"
ajdodlar = nx.ancestors(G, nishon)
print(f"\n{nishon} ga olib kelgan obyektlar: {', '.join(sorted(ajdodlar))}")
hujjat = "C:/Users/aziz/Downloads/hisobot.docm"
print("  zanjir:", " -> ".join(nx.shortest_path(G, hujjat, nishon)))


203.0.113.50:8443 ga olib kelgan obyektlar: C:/Users/aziz/Downloads/hisobot.docm, explorer.exe, outlook.exe, powershell.exe, upd.exe, winword.exe
  zanjir: C:/Users/aziz/Downloads/hisobot.docm -> winword.exe -> powershell.exe -> upd.exe -> 203.0.113.50:8443


In [6]:
# 5. Oldinga kuzatish: zararli hujjat yana nimalarga ta'sir qilgan?
print("\nHujjat ochilgandan keyin ta'sirlangan obyektlar:")
for obyekt in sorted(nx.descendants(G, hujjat)):
    print("  ", obyekt)


Hujjat ochilgandan keyin ta'sirlangan obyektlar:
   198.51.100.23:443
   203.0.113.50:8443
   C:/Users/aziz/AppData/upd.exe
   powershell.exe
   upd.exe
   winword.exe


## Kitobdagi natija

Dasturni ishga tushirganingizda quyidagiga o'xshash natija chiqishi kerak (tasodifiy sonlar generatori urug'i qat'iy belgilangani uchun natija odatda aynan bir xil bo'ladi; faqat bajarilish vaqti va har safar yangi yaratiladigan tasodifiy kalitlar farq qilishi mumkin):

```
Shubhali jarayon juftliklari:
  09:06  winword.exe -> powershell.exe   (30 kunda 0 marta)
  09:07  powershell.exe -> upd.exe   (30 kunda 0 marta)

203.0.113.50:8443 ga olib kelgan obyektlar: C:/Users/aziz/Downloads/hisobot.docm, explorer.exe, outlook.exe, powershell.exe, upd.exe, winword.exe
  zanjir: C:/Users/aziz/Downloads/hisobot.docm -> winword.exe -> powershell.exe -> upd.exe -> 203.0.113.50:8443

Hujjat ochilgandan keyin ta'sirlangan obyektlar:
   198.51.100.23:443
   203.0.113.50:8443
   C:/Users/aziz/AppData/upd.exe
   powershell.exe
   upd.exe
   winword.exe
```

## O'zingiz sinab ko'ring

Parametrlarni (ma'lumotlar hajmi, chegara qiymati, tasodifiy son urug'i) o'zgartirib, natija qanday o'zgarishini kuzating. Kitobdagi 2-bobning B darajadagi topshiriqlari aynan shu daftar asosida bajariladi.